In [3]:
import numpy as np
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_1_1.csv')

# print(df.info())
# print(df.head())

df1 = df.groupby('소주제')['정답여부'].sum() / df.groupby('소주제')['정답여부'].count()
# print(help(df1.sort_values(ascending=False).rank))
print(df1[df1.sort_values(ascending=False).rank(ascending=False, method='dense')==3])

소주제
경제    0.68
Name: 정답여부, dtype: float64


In [77]:
import numpy as np
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_1_2.csv')

# print(df.info())
# print(df.head())

df['연도'] = df['date'].str.split('-').str[0]
df['월'] = df['date'].str.split('-').str[1]
df1 = df.groupby(['연도','월'])['price'].sum()
print(df1.sort_values(ascending=False).iloc[1])
df2 = df[(df['연도']==df1[df1==df1.sort_values(ascending=False).iloc[3]].index[0][0]) & (df['월']==df1[df1==df1.sort_values(ascending=False).iloc[3]].index[0][1])].groupby('category')['price'].sum()
print(df2.sort_values(ascending=False).iloc[0])

1777389
1012500


In [108]:
import numpy as np
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_1_3.csv')

# print(df.info())
# print(df.head())

df['split'] = df['message'].str.split(' ')
df['split_count'] = df['split'].apply(len)
df1 = df.groupby('label')['split_count'].mean()
print(np.round(abs(df1.loc['ham'] - df1.loc['spam']),3))

0.3


In [1]:
import numpy as np
import pandas as pd

train_df = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_2_train.csv')
test_df = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_2_test.csv')

print(train_df.info())
print(train_df.head())
print(test_df.info())
print(test_df.head())

train_X = train_df.drop('gas_totl', axis=1)
train_y = train_df['gas_totl']
test_X = test_df.drop('gas_totl', axis=1)
test_y = test_df['gas_totl']

# print(train_X.shape, train_y.shape)

from sklearn.model_selection import train_test_split
train_X, valid_X, train_y, valid_y = train_test_split(train_X, train_y, test_size=0.3)
# print(train_X.shape, valid_X.shape, train_y.shape, valid_y.shape)

cat_X = train_X.select_dtypes('object').columns.to_list()
num_X = train_X.select_dtypes('number').columns.to_list()

from sklearn.preprocessing import OneHotEncoder
onehotencoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
one_train_X = onehotencoder.fit_transform(train_X[cat_X])
one_valid_X = onehotencoder.transform(valid_X[cat_X])
one_test_X = onehotencoder.transform(test_X[cat_X])

train_X1 = np.concatenate([train_X[num_X], one_train_X], axis=1)
valid_X1 = np.concatenate([valid_X[num_X], one_valid_X], axis=1)
test_X1 = np.concatenate([test_X[num_X], one_test_X], axis=1)

from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor()
rf.fit(train_X1, train_y)

pred1 = rf.predict(valid_X1)
pred2 = rf.predict(test_X1)

from sklearn.metrics import root_mean_squared_error
print(root_mean_squared_error(valid_y, pred1))

pred2 = pd.DataFrame({'pred': pred2})
pred2.to_csv('result.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   biz_type        160 non-null    object 
 1   area            160 non-null    float64
 2   age             160 non-null    int64  
 3   num_households  160 non-null    int64  
 4   gas_totl        160 non-null    int64  
dtypes: float64(1), int64(3), object(1)
memory usage: 6.4+ KB
None
      biz_type       area  age  num_households  gas_totl
0       public  93.291780   33              61       499
1  residential  97.509495   34             131      1156
2  residential  97.140181    5              30       392
3  residential  65.789408   12             166      1343
4        mixed  92.631000   28              14       154
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          ----

In [5]:
import numpy as np
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_3_1.csv')

# print(df.info())
# print(df.head())

import statsmodels.api as sm
import statsmodels.formula.api as smf
model = smf.glm("attrition ~ age + income + overtime", data=df, family=sm.families.Binomial()).fit()
# print(model.summary())
df1 = model.pvalues[1:] < 0.05
df2 = model.params
rs = df1[df1==True].index[0]
print(df2[df2.index==rs].round(3).iloc[0])

print(np.exp(df2.loc['age']).round(3))

# df.info()
test = pd.DataFrame({'age': [20], 'income': [3000], 'overtime': [2]})

pred = model.predict(test)
print(pred.round(3)[0])

-0.005
0.905
0.431


In [52]:
import numpy as np
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_3_2.csv')

# print(df.info())
# print(df.head())

import statsmodels.formula.api as smf
model = smf.ols("price ~ area + height + wall", data=df).fit()
print(model.summary())

df1 = model.pvalues[1:] < 0.05
# df1[df1==True].index
df2 = model.params
print(df2[df1[df1==True].index].sum().round(3))

model1 = smf.ols("price ~ area + height", data=df).fit()
print(model1.rsquared.round(3))

test = pd.DataFrame({'area': [100],
                     'height': [10],
                     'wall': [1]})
print(model1.predict(test).round(3)[0])

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.859
Model:                            OLS   Adj. R-squared:                  0.857
Method:                 Least Squares   F-statistic:                     399.4
Date:                Fri, 28 Nov 2025   Prob (F-statistic):           3.29e-83
Time:                        14:39:28   Log-Likelihood:                -885.10
No. Observations:                 200   AIC:                             1778.
Df Residuals:                     196   BIC:                             1791.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      2.3889      9.460      0.253      0.8